# Упражнение 2: данни, подготовка и мащабиране

**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Цел на упражнението](#cel)
2. [Файлове в упражнението](#failove)
3. [Подготовка и стартиране](#podgotovka)
4. [Част 1. pandas: таблици и оглед](#chast-1)
5. [Част 2. scikit-learn: подготовка на данните](#chast-2)
6. [Част 3. scikit-learn: мащабиране](#chast-3)
7. [Част 4. Как мащабът променя сходството](#chast-4)

<a id="cel"></a>
## Цел на упражнението

В първото упражнение работихме с малка таблица и един признак. Сега ще
разгледаме таблица със 150 ириса. За всяко цвете имаме четири числови
измервания, категориен признак `light_conditions` и вид `species`,
който ще използваме като цел. `light_conditions` описва условията на
осветяване: "слънце", "полусянка" или "сянка".

Колоната за осветяване е **измислена за упражнението**. Оригиналният
набор Iris не съдържа такива сведения. Стойностите са разпределени
между редовете само за да упражним работа с категории; те не са реални
наблюдения и не са изчислени от четирите измервания.

Ще проверим формата, типовете, разпределението по категории и липсващите
стойности. После ще отделим данни за проверка, ще попълним пропуските,
ще преобразуваме категориите в числови колони и ще мащабираме измерванията.

**pandas** ни дава колони с имена и удобни операции за оглед и избор на
редове. **scikit-learn** ни дава преобразуватели, които научават нужните
стойности от тренировъчните данни и прилагат същото правило към нови редове.
Накрая ще видим как мащабирането може да промени най-близкия съсед.

<a id="failove"></a>
## Файлове в упражнението

Пътищата са относителни спрямо папката на тази тетрадка.

| Файл | За какво служи |
|---|---|
| `Упражнение_2.ipynb` | обяснения, примери и задачи |
| `ЗА_ВКЪЩИ.ipynb` | задачи за самостоятелна работа |
| `data/iris_with_missing.csv` | 150 ириса с четири измервания, изкуствено добавени условия на осветяване и вид |
| `../environment.yml` | обща среда при работа с `conda` |
| `../requirements.txt` | общ списък с пакети при работа с `pip` |

Файлът е учебен вариант на набора Iris от scikit-learn. Шест измервания
са изтрити нарочно. Колоната `light_conditions` е измислена и съдържа
циклично разпределени стойности "слънце", "полусянка" и "сянка".
Оригиналните данни не включват условия на осветяване. Празната клетка
означава липсващо измерване, а не нула. Файлът е локален.

<a id="podgotovka"></a>
## Подготовка и стартиране

Използваме средата, създадена по време на първото упражнение. От папката на **второто**
упражнение обновяваме пакетите по избрания начин (само при нужда, но е хубаво да имаме този навик).
ВАЖНО: обновяването се прави след като сме избрали виртуалната реда, не преди това:

| Среда | Команди |
|---|---|
| Anaconda Prompt | 1. `conda activate ml2627`<br/>2. `conda env update -f ../environment.yml`  |
| Чрез `pip` | 1. `..\venv\Scripts\activate.bat`<br/>2. `python -m pip install -r ../requirements.txt` |

След това, от същата папка, стартираме `jupyter lab`, отваряме
`Упражнение_2.ipynb` и избираме ядрото **Python (МС 26/27)**.
Ако даден пакет се намира в терминала, но липсва в тетрадката,
проверяваме избраното ядро. Ако CSV файлът не се намира, проверяваме
от коя папка сме стартирали Jupyter.

In [ ]:
import os
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("работна папка:", os.getcwd())

<a id="chast-1"></a>
## Част 1. pandas: таблици и оглед

**pandas** е библиотека за таблични данни. Основният й обект е
`DataFrame`: таблица, в която редовете са примери, колоните имат имена
и всяка колона може да има свой тип. Отделна колона е `Series`.
За разлика от обикновен NumPy масив, тук използваме имена на колони
и индекс на редовете, за да избираме данните по смисъл.

С pandas ще прочетем CSV файла, ще проверим какви данни имаме, ще
открием пропуските и ще изберем редовете и колоните, които ни трябват.

### 1. Четем таблицата

`pd.read_csv` прочита файла, като използва първия ред за имена на
колоните. Празните полета стават липсващи стойности. `df.shape` връща
двойката (брой редове, брой колони), а `df.head()` избира първите пет
реда. Това е бърз начин да проверим дали файлът е прочетен правилно.

`display` идва от `IPython.display`; импортирахме го в началната кодова
клетка. В тетрадка то показва таблицата във вид, удобен за четене.
`print` показва текстовото представяне. Нито едно от двете не променя
данните.

Всеки ред е едно цвете. Четирите измервания са числови признаци;
`light_conditions` е измислен категориен признак за условията на
осветяване, а `species` е целта `y`, която описва вида.

In [ ]:
df = pd.read_csv("data/iris_with_missing.csv")
print("Форма (редове, колони):", df.shape)
display(df.head())

### 2. Една колона като `Series` или `DataFrame`

`df["sepal_length_cm"]` връща `Series` с една ос: 150 стойности.
`df[["sepal_length_cm"]]` връща `DataFrame` с две оси: 150 реда и
една колона. Външните двойни скоби означават, че подаваме списък
от имена на колони. Тази разлика има значение, когато функция
очаква таблица, а не единична поредица от стойности.

In [ ]:
length_series = df["sepal_length_cm"]
length_table = df[["sepal_length_cm"]]

print("Series:", type(length_series).__name__, length_series.shape)
print("DataFrame:", type(length_table).__name__, length_table.shape)
display(length_series.head(3))
display(length_table.head(3))

### 3. Избираме редове по етикет или позиция

#### `df.index`: етикетите на редовете
`df.index` показва с какви етикети са означени редовете. При новосъздадена таблица те често са 0, 1, 2, 3, ... Етикетът обаче не е задължително да съвпада с текущата позиция на реда.
Например след филтриране може да останат три реда с етикети 0, 2, 5. Те вече са на позиции 0, 1, 2 в получената таблица, но етикетите им остават 0, 2, 5.

#### `df.loc`: избор по етикет
С `df.loc` посочваме етикет на ред и, ако е нужно, име на колона. При диапазон с `df.loc` крайният етикет се включва. Затова `df.loc[0:2]` включва реда с етикет 2.

#### `df.iloc`: избор по позиция
С `df.iloc` броим редовете и колоните от нула според мястото им в таблицата. Тук важи обичайното правило за срезовете в Python:
крайната позиция не се включва. Затова `0:3` означава позиции 0, 1 и 2.

In [ ]:
print("Имена на колоните:", df.columns.tolist())
print("Първи индекси:", df.index[:5].tolist())

display(df.loc[0:2, ["sepal_length_cm", "light_conditions", "species"]])
display(df.iloc[0:3, [0, 4, 5]])
df_reversed = df.iloc[::-1] # [начало:край:стъпка]
display(df_reversed.loc[2:0, ["sepal_length_cm", "light_conditions", "species"]]) 
display(df_reversed.iloc[0:3, [0, 4, 5]])

Какво ще изведе следния код?

`display(df_reversed.loc[2:0, ["sepal_length_cm", "light_conditions", "species"]])`

In [ ]:
# За да направим дадена колона да е индексна:
df_by_species = df.set_index("species")
display(df_by_species.loc["setosa"])

### 4. Проверяваме типовете, пропуските и категориите

`df.dtypes` показва типа на всяка колона. Числовите измервания са
`float64`, а текстовите категории са `object`. `isna()` връща
`True` за всяка липсваща клетка; `sum()` преброява тези стойности
по колони. `value_counts()` брои срещанията на всяка категория.

`light_conditions` и `species` са **номинални категории**: стойностите
им нямат естествен числов ред. "слънце", "полусянка" и "сянка" са
наименования на условия, а не числови измервания. `species` е цел,
а не признак за предсказването й.

In [ ]:
print("Типове:")
print(df.dtypes)
print()
print("Липсващи стойности по колони:")
print(df.isna().sum())
print()
print("Цветя по вид:")
print(df["species"].value_counts())
print()
print("Цветя по условия на осветяване:")
print(df["light_conditions"].value_counts().sort_index())

### 5. Обобщаваме числовите колони

`describe()` изчислява по колони броя налични стойности, средното,
стандартното отклонение, квартилите, минимума и максимума. `.T`
обръща таблицата, за да е по-лесно да сравним четирите измервания;
`round(2)` закръгля показаните резултати до два знака. `count` е под
150 там, където има липсващи измервания.

С `groupby("species")` разделяме редовете по вид и изчисляваме
медианата за всяка група.

In [ ]:
numeric_features = [
    "sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"
]
categorical_features = ['light_conditions']
feature_names = numeric_features + categorical_features

display(df[numeric_features].describe().T.round(2))
display(df.groupby("species")[numeric_features].median().round(2))

### 6. Избираме редове по условие

Сравнението `df["species"] == "virginica"` дава булева `Series`:
`True` за подходящите редове и `False` за останалите. С `&`
съединяваме две условия, като ограждаме всяко в скоби. После
`df.loc[mask, columns]` избира редовете с `True` и посочените колони.

In [ ]:
mask = (df["species"] == "virginica") & (df["petal_length_cm"] > 5)
selected = df.loc[mask, ["petal_length_cm", "petal_width_cm", "species"]]
print("Избрани редове:", len(selected))
display(selected.head())

#### Нека опитаме 1

Избираме `versicolor` с `sepal_width_cm` под 3. Показваме двете
колони `sepal_width_cm` и `species` и броя на избраните редове.

<details><summary>Подсказка</summary>

Съединяваме две условия с `&`. Използваме `df.loc[mask, columns]`.

</details>

In [ ]:
# selected = ...
# display(selected)
# print(len(selected))

### 7. Гледаме две измервания заедно

Точките са цветя, а цветовете показват вида. Диаграмата ни помага
да видим как се разполагат измерванията, но сама по себе си не е модел.
Ред с липсваща стойност по някоя от двете оси не се вижда на графиката.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
for species, group in df.groupby("species"):
    ax.scatter(group["petal_length_cm"], group["petal_width_cm"],
               label=species, alpha=0.75)
ax.set(xlabel="дължина на венчелистче (см)",
       ylabel="ширина на венчелистче (см)",
       title="Ирисите по два признака")
ax.legend(title="вид")
plt.show()

<a id="chast-2"></a>
## Част 2. scikit-learn: подготовка на данните

**scikit-learn** предлага готови инструменти за разделяне на данни,
подготовка на признаци и обучение на модели. Тук използваме
**преобразуватели**: `SimpleImputer` за пропуските и `OneHotEncoder`
за номиналния признак. В следващата част ще използваме и преобразуватели
за мащабиране. Всички работят по сходен начин:

| Метод | Какво прави в нашия пример |
|---|---|
| `fit(тренировъчни_данни)` | научава медиани, категории или мащаб от тренировъчните редове |
| `transform(данни)` | прилага наученото правило към тренировъчни или тестови редове |
| `fit_transform(тренировъчни_данни)` | съчетава двете стъпки за тренировъчните редове |

Първо отделяме тестовите редове. После извикваме `fit` само върху
тренировъчните, за да не повлияят тестовите данни на подготовката. `transform`
връща NumPy масив. Когато искаме да запазим имената на колоните
и индексите, го обвиваме отново в pandas `DataFrame`.

### 8. Намираме редовете с пропуски

Липсващата стойност не е нула. Заместването с нула би добавило
измислено измерване. Първо виждаме кои редове са засегнати.
Малко по-долу ще видим как да процедираме с тях.

In [ ]:
rows_with_missing = df.loc[df[numeric_features].isna().any(axis=1)]
print("Редове с поне един пропуск:", len(rows_with_missing))
display(rows_with_missing)

### 9. Отделяме признаците, целта и тестовите редове

`X` трябва да съдържа четирите числови измервания и условията на осветяване.
`y` трябва да съдържа вида на цветето. За обучение и проверка разделяме редовете
**преди** да научим медиани, категории или параметри за мащабиране.
Не можем обаче да направим разлелението като вземем първите `N` реда за тренировъчно множество.
Трябва да имаме различни примери при обучението и при тестването. Един вариант е да се разделят
на случаен принцип. Друг чрез `stratify=y`, който запазва приблизително еднакъв дял от трите вида цвета в двете части.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_names]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Tренировъчни:", X_train.shape, "Тестови:", X_test.shape)
print("Пропуски в тренировъчните:", int(X_train.isna().sum().sum()))
print("Пропуски в тестовите:", int(X_test.isna().sum().sum()))

### 10. Попълваме липсващите измервания

За всяка числова колона `SimpleImputer(strategy="median")` намира
медиана от **тренировъчните** редове. `fit` научава четири медиани, а
`transform` ги прилага и към тренировъчните, и към тестовите редове.
Запазваме имената на колоните и първоначалните индекси в нови таблици.
Категорийният признак не участва в изчисляването на медианите.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median").fit(X_train[numeric_features])

X_train_filled = pd.DataFrame(
    imputer.transform(X_train[numeric_features]),
    index=X_train.index, columns=numeric_features
)

X_test_filled = pd.DataFrame(
    imputer.transform(X_test[numeric_features]),
    index=X_test.index, columns=numeric_features
)

print("Научени медиани:")
print(pd.Series(imputer.statistics_, index=numeric_features).round(2))
print("Останали без стойност:",
      int(X_train_filled.isna().sum().sum()),
      int(X_test_filled.isna().sum().sum()))

> **Въпрос.** Каква е разликата между двата сегмента код?

```python
X_train_filled = pd.DataFrame(
    imputer.transform(X_train[numeric_features]),
    index=X_train.index, columns=numeric_features
)
```

```python
X_train_filled = X_train.copy()
X_train_filled[numeric_features] = imputer.transform(
    X_train[numeric_features]
)
```

<details><summary>Отговор</summary>

В първия сегмент губим `light_conditions` колоната.

</details>

| Стратегия | С какво попълва | Подходяща употреба |
|---|---|---|
| `"median"` | Медианата | Числови данни с крайни стойности |
| `"mean"` | Средното аритметично | Числови данни без силни крайни стойности |
| `"most_frequent"` | Най-често срещаната стойност | Често за категории като `light_conditions` |
| `"constant"` | Стойност, зададена чрез `fill_value` | Например категория `"неизвестно"` или число `0` |

#### Нека опитаме 2

Виждаме първия тренировъчен ред, в който е имало липсващо **измерване**,
преди и след попълването. Коя стойност е заменена? Използваме индекса
на реда, за да сравним същото цвете в двете таблици.

<details><summary>Подсказка</summary>

`X_train.index[X_train[numeric_features].isna().any(axis=1)][0]` дава
индекса. После използваме `.loc[index]` за двата реда.

</details>

In [ ]:
# missing_index = ...
# display(X_train.loc[missing_index])
# display(X_train_filled.loc[missing_index])

### 11. Кодираме категорийния признак

Моделът работи с числа, но не можем просто да заменим "слънце",
"полусянка" и "сянка" с `1`, `2`, `3`: така бихме внушили ред и
разстояние между условията. При **one-hot кодиране** създаваме по една
колона за всяка категория. Ако стойността е "полусянка", колоната
`light_conditions_полусянка` става 1, а другите две индикаторни
колони са 0.

`OneHotEncoder` научава възможните категории само от **тренировъчните** редове.
С `handle_unknown="ignore"` можем да преобразуваме и нова, непозната
категория; тогава всички нейни индикаторни колони са 0. Тук няма
липсващи стойности в `light_conditions`.

In [ ]:

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train[categorical_features])
encoded_names = encoder.get_feature_names_out(categorical_features)
X_train_encoded = pd.DataFrame(
    encoder.transform(X_train[categorical_features]),
    index=X_train.index, columns=encoded_names
)
X_test_encoded = pd.DataFrame(
    encoder.transform(X_test[categorical_features]),
    index=X_test.index, columns=encoded_names
)

print("Научени категории:", encoder.categories_[0].tolist())
display(pd.concat([X_train[categorical_features].head(5),
                   X_train_encoded.head(5)], axis=1))


> **Въпрос.** Какво ще получим за нова стойност "изкуствена светлина"?
> Защо нулите не означават, че тя е еднаква с някое от трите познати
> условия на осветяване?

```python
encoder.transform(pd.DataFrame({"light_conditions": ["изкуствена светлина"]}))
```

<details><summary>Отговор</summary>

Получаваме `[[0., 0., 0.]]`. "Изкуствена светлина" не присъства сред
категориите, научени от тренировъчните редове. Нулите само показват, че тя
не съвпада с нито една от тях. Кодиращият преобразувател не знае колко
е сходна с тях.

</details>

<a id="chast-3"></a>
## Част 3. scikit-learn: мащабиране

### 12. Защо мащабираме

Някои модели сравняват цветята чрез разстояние между стойностите на признаците им.
Тогава колоните с по-големи числови разлики могат да имат по-силно влияние върху резултата.

Например при сравнение на две цветя разликата в един признак може да е 1 cm, а в друг — 0,1 cm.
При изчисляване на разстоянието първата разлика ще има много по-голям принос.
Това може да се случи и тук: четирите измервания са в сантиметри, но диапазоните и изменчивостта
им са различни. **Мащабирането** преобразува числовите колони така, че разликите между тях да станат
по-съпоставими.

Ще сравним два начина за мащабиране, които се прилагат поотделно за всяка числова колона:
- **Min-max мащабиране**: от всяка стойност изваждаме минимума на тренировъчната колона, след което делим на диапазона ѝ - максимума минус минимума: $$x_{\text{ново}}=\frac{x-\min}{\max-\min}$$ Така най-малката тренировъчна стойност става 0, а най-голямата - 1. Останалите тренировъчни стойности попадат между тях.
- **Стандартизация**: от всяка стойност изваждаме средната стойност на тренировъчната колона и делим на стандартното ѝ отклонение: $$z=\frac{x-\mu}{\sigma}$$ Получената тренировъчна колона има средна стойност около 0 и стандартно отклонение около 1. Стандартизираните стойности не са ограничени между 0 и 1.

И при двата метода изчисляваме необходимите стойности чрез fit само от тренировъчните данни. После използваме същото преобразуване за тестовите данни.

Индикаторните колони от one-hot кодирането вече съдържат 0 и 1, затова в този пример няма да ги мащабираме. species е целевата колона и не участва в преобразуването на признаците.

### 13. Min-max мащабиране

`MinMaxScaler` използва минимума и максимума на **попълнените тренировъчни**
колони. Тренировъчниите стойности попадат между 0 и 1. При тестовите това
не е гарантирано: ново измерване може да е извън тренировъчния диапазон.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler().fit(X_train_filled)
X_train_minmax = pd.DataFrame(
    minmax_scaler.transform(X_train_filled),
    index=X_train.index, columns=numeric_features
)
X_test_minmax = pd.DataFrame(
    minmax_scaler.transform(X_test_filled),
    index=X_test.index, columns=numeric_features
)
print("Тренировъчни минимуми:", X_train_minmax.min().round(2).to_dict())
print("Тренировъчни максимуми:", X_train_minmax.max().round(2).to_dict())
print("Най-малка тестова стойност:", round(X_test_minmax.min().min(), 3))

> **Въпрос.** Какво ще отпечата този нов код? Защо резултатът може
> да е под нула, въпреки че използваме `MinMaxScaler`?

```python
scaler = MinMaxScaler().fit([[2.0], [4.0]])
print(scaler.transform([[1.0]]))
```

<details><summary>Отговор</summary>

`[[-0.5]]`. Тренировъчният минимум е 2, а новата стойност е 1. Използваме
същото правило `(1 - 2) / (4 - 2)`, без да променяме научения минимум.

</details>

### 14. Стандартизация

`StandardScaler` научава средно и стандартно отклонение за всяка
попълнена колона. Прилагаме наученото преобразуване и към
тестовите редове. Сравняваме първите три тренировъчни реда преди и след.

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler().fit(X_train_filled)
X_train_scaled = pd.DataFrame(
    standard_scaler.transform(X_train_filled),
    index=X_train.index, columns=numeric_features
)
X_test_scaled = pd.DataFrame(
    standard_scaler.transform(X_test_filled),
    index=X_test.index, columns=numeric_features
)

print("Средни след стандартизация:", X_train_scaled.mean().round(2).to_dict())
print("Стандартни отклонения:", X_train_scaled.std(ddof=0).round(2).to_dict())
display(X_train_filled.head(3))
display(X_train_scaled.head(3).round(2))

Не очакваме средното на **тестовите** редове да е точно нула.
Използваме параметрите от тренировъчните редове, а тестовите имат свои
стойности. Ако извикаме `fit` върху тестовите редове, ще променим
правилото и ще използваме информация от набора за проверка.

### 15. Събираме подготвените признаци

Числовите и категорийният признак са подготвени по различен начин.
Събираме стандартизираните измервания и one-hot колоните в една числова
таблица. Индексите пазят съответствието между редовете при `concat`.
За сравнение ще съберем и попълнените, но немащабирани измервания със
същите one-hot колони.

Условията на осветяване са измислени за упражнението. Наличието им в
готовата таблица не доказва, че помагат за предсказване на вида.

In [ ]:

X_train_scaled_ready = pd.concat([X_train_scaled, X_train_encoded], axis=1)
X_test_scaled_ready = pd.concat([X_test_scaled, X_test_encoded], axis=1)
X_train_minmax_ready = pd.concat([X_train_minmax, X_train_encoded], axis=1)
X_test_minmax_ready = pd.concat([X_test_minmax, X_test_encoded], axis=1)

print("Готови тренировъчни и тестови таблици с стандартизиране:", X_train_scaled_ready.shape, X_test_scaled_ready.shape)
display(X_train_scaled_ready.head(3).round(2))

print("Готови тренировъчни и тестови таблици със minmax мащабиране:", X_train_minmax_ready.shape, X_test_minmax_ready.shape)
display(X_train_minmax_ready.head(3).round(2))


<a id="chast-4"></a>
## Част 4. Как мащабът променя сходството

### 16. Сравняваме най-близките съседи

Нека си представим, че имаме нов ирис и искаме да намерим **най-подобното цвете** сред тренировъчните данни. 
За целта сравняваме неговите признаци с признаците на всеки тренировъчен ирис. Цветето с най-малко
разстояние до новия ирис наричаме негов **най-близък съсед**. По този начин се надяваме, че ще определим точно вида му.
Имаме 4 тестови примера, които не са използвани за "обучението" и може да се провери с тях.

Ще измерваме близостта с **евклидово разстояние**. То събира разликите по отделните признаци в едно число:
по-малко разстояние означава по-близки стойности на признаците. За всеки от 30-те тестови ириса ще изчислим
разстоянието до всички тренировъчни ириси и ще намерим тренировъчния ред с най-малко разстояние.

Ирисите имат четири числови измервания и one-hot колона за условията на осветяване. Ще направим сравнението
два пъти: веднъж след min-max мащабиране и веднъж след стандартизация. **Тестовите и тренировъчните цветя 
остават същите**, както и one-hot колоните. Променя се само начинът, по който са мащабирани четирите измервания.

Защо най-близкият съсед може да се промени? Евклидовото разстояние зависи от числовите разлики по всички колони.
Двата метода за мащабиране променят относителния принос на измерванията към това разстояние.
Затова едно тренировъчно цвете може да се окаже най-близко при min-max мащабиране, а друго - при стандартизация.
Сравнението ще ни покаже **дали и за кои тестови ириси** се случва това.

**Въпрос**: Очакваме ли най-близкият тренировъчен ред да остане същият за всичките 30 тестови ириса? Защо?
<details><summary>Отговор</summary>
Отговорът: не е задължително. При min-max мащабиране и стандартизация разликите по четирите измервания получават различна тежест в евклидовото разстояние. Затова най-близкият тренировъчен ирис може да се смени. Възможно е обаче и за четирите тестови ириса да остане същият - това ще установим едва след изчислението.
</details>

In [ ]:
from sklearn.metrics import pairwise_distances

minmax_distances = pairwise_distances(X_test_minmax_ready, X_train_minmax_ready)
scaled_distances = pairwise_distances(X_test_scaled_ready, X_train_scaled_ready)
minmax_nearest = minmax_distances.argmin(axis=1)
scaled_nearest = scaled_distances.argmin(axis=1)

print("Тестови цветя с променен най-близък тренировъчен ред:",
      int(np.count_nonzero(minmax_nearest != scaled_nearest)), "от", len(X_test))
for i in range(len(X_test)):
    print("Тестов ирис, индекс в CSV:", X_test.index[i])
    print("- най-близък по minmax:", X_train.index[minmax_nearest[i]])
    print("- най-близък по стандартизация:", X_train.index[scaled_nearest[i]])

Различният най-близък ред показва, че мащабирането променя смисъла
на разстоянието. Кодирането на категорийния признак също задава начин
за измерване на разлика между условията на осветяване. Това **не доказва** само по
себе си, че някоя от двете версии е по-подходяща за предсказване.

<a id="zapomnyane"></a>
## За запомняне

1. pandas представя таблицата като `DataFrame`, а отделната колона като `Series`.
   Редът е пример. Колоните имат тип и смисъл. `species` е категорийна цел,
   а `light_conditions` е измислен за упражнението категориен признак.
2. Преди подготовка проверяваме форма, типове, категории и клетки без данни.
3. Отделяме тестовите редове, преди да научим медиани, категории и мащаб.
4. `fit` научава параметри от тренировъчните данни. `transform` ги прилага
   към тренировъчни и тестови редове.
5. Попълваме и мащабираме **числовите** колони. Номиналните категории
   преобразуваме с one-hot кодиране.
6. Изборът на представяне може да промени разстоянията и най-близкия съсед.